# 04 — Quantum Mutual Information on Real Hardware

In `02/10` a Bell pair shared exactly 2 bits of mutual information — double the classical maximum any pair of bits can reach. That factor of two *is* entanglement, written in the language of information. Here you measure that same quantity on a real superconducting chip and watch decoherence quietly steal a piece of it.

**Prerequisites:** `01/10_entanglement`, `02/10_quantum_mutual_information`, `05/03_tomography_and_fidelity_on_real_hardware`.

**What you'll be able to do**
- Prepare a Bell pair on the device and run full two-qubit tomography (nine measurement settings).
- Reconstruct the joint state `rho_AB` from the device's counts with `density_from_counts`.
- Compute the quantum mutual information `I(A:B)` and watch it fall below the ideal 2 bits — below it because of decoherence *and* finite-shot effects together.

You have already reconstructed a single-qubit state on hardware in `05/03` and read its fidelity gap. Now you lift that machinery to two qubits and recover the *joint* state of a Bell pair — the first information-geometric quantity of this module to meet a live device. Measuring entanglement on a chip cooled to a few millikelvin, from nothing but click statistics, is a genuine milestone. Let us go and do it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.runtime import select_backend
from qot_course.hardware.tomography import two_qubit_tomography_circuits, density_from_counts
from qot_course.infotheory.quantum import quantum_mutual_information
from qot_course.quantum.composite import bell_state
from qot_course.quantum.density import density_matrix, fidelity

np.random.seed(42)
viz.use_course_style()

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")

## 1 · Two-qubit tomography

One qubit needed three measurement settings — one per Pauli axis. Two qubits need the nine combinations of $\{X, Y, Z\}$ on each: $XX, XY, XZ, YX, \ldots, ZZ$. Together those nine settings pin down every entry of the $4 \times 4$ joint density matrix, exactly as the three single-qubit settings fixed the Bloch vector in `05/03`. We reuse the very same reconstruction, `density_from_counts`, now told there are two qubits.

From the reconstructed `rho_AB` we compute the quantum mutual information

$$I(A:B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB}),$$

where $S(\rho) = -\mathrm{Tr}(\rho \log_2 \rho)$ is the von Neumann entropy in bits and $\rho_A, \rho_B$ are the reduced states of each qubit. For a pure Bell pair each qubit alone is maximally mixed ($S = 1$ bit) while the joint state is pure ($S = 0$), so $I(A:B) = 1 + 1 - 0 = 2$ bits — the ceiling a real device cannot quite reach.

In [ ]:
prep = QuantumCircuit(2)
prep.h(0)
prep.cx(0, 1)                               # Bell pair (|00> + |11>)/sqrt(2)
circuits = two_qubit_tomography_circuits(prep)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
settings = list(circuits)                   # 9 (basis_q0, basis_q1) tuples
isa_list = [pm.run(circuits[s]) for s in settings]
res = SamplerV2(mode=backend).run(isa_list, shots=4096).result()
counts_by_setting = {s: res[i].data.c.get_counts() for i, s in enumerate(settings)}

rho_ab = density_from_counts(counts_by_setting, n_qubits=2)
ideal = density_matrix(bell_state())
qmi_device = quantum_mutual_information(rho_ab, dims=[2, 2])   # bits
qmi_ideal = quantum_mutual_information(ideal, dims=[2, 2])
print(f"fidelity(rho_AB, Bell) = {fidelity(rho_ab, ideal):.3f}")
print(f"I(A:B) ideal  = {qmi_ideal:.3f} bits")
print(f"I(A:B) device = {qmi_device:.3f} bits   (below the ideal 2 bits)")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.bar(["ideal Bell", "device"], [qmi_ideal, qmi_device],
       color=[COLORS["muted"], COLORS["quantum"]], alpha=0.9)
ax.axhline(1.0, color=COLORS["highlight"], linestyle="--", label="classical maximum (1 bit)")
ax.set_ylabel("I(A:B)  (bits)")
ax.set_title("Quantum mutual information: ideal Bell pair vs a real device", pad=12)
ax.legend()
plt.show()

**Read the figure.** The ideal pair clears the classical 1-bit ceiling and reaches 2 bits — the information signature of maximal entanglement. (That reading is exact here because the Bell pair is a *pure* state: for a pure state, quantum mutual information equals twice the entanglement entropy, so the 2-bit ceiling is genuinely a statement about entanglement. For a general mixed state, QMI also counts classical correlations and is not an entanglement measure on its own — a distinction that matters once the device has already introduced noise into `rho_AB`.) The device bar falls short. That shortfall is *mostly* decoherence: every CX and every read-out error leaks a little of the correlation that the Bell pair stores. But part of the gap is not decoherence at all — it is finite-shot tomography plus the projection back to a physical state inside `density_from_counts`. Even a noiseless device at 4096 shots per setting would land a touch below 2 bits, because sampling noise and the eigenvalue-clip projection each bias the reconstructed entropy. Raising the shot count shrinks *that* part; the remaining, irreducible gap is the device's true entanglement loss. Offline this reflects the modeled FakeManila noise; flip `USE_HARDWARE=True` and the very same bar reports the literal chip's gap.

### A real run on IBM hardware

This notebook was run once on a real quantum computer. On **`ibm_marrakesh`** (156-qubit Heron processor, 2026-05-30, job `d8dih47d0j8c73f4apmg`), the full nine-setting two-qubit tomography returned:

| quantity | ideal | this device |
|---|---|---|
| fidelity to the Bell state | 1.00 | **0.96** |
| I(A:B) | 2.00 bits | **1.74 bits** |

A genuine measurement: `I(A:B) = 1.74` bits sits below the ideal 2 (decoherence has taken a share of the correlation) while still clearing the classical 1-bit ceiling. A modern device improves on the offline FakeManila noise model the default switch uses (about 1.38 bits). Set `USE_HARDWARE = True` to reproduce on the least-busy QPU your account can reach.

## 2 · Why entanglement is fragile

The CX (entangling) gate and read-out are the noisiest operations on a superconducting device — far noisier than the single-qubit rotations that build `|+>`. A Bell pair is *made of* exactly those operations: a Hadamard followed by a CX. So the correlations that make a Bell pair special are precisely what decoheres first, which is why `I(A:B)` is a more demanding probe of a device than the single-qubit fidelity of `05/03`.

This extends the classical-to-quantum dictionary one more row, in the same spirit as the `ideal -> device` rows you built in `05/03`:

| Ideal object | What a real device returns |
|---|---|
| Bell pair with `I(A:B) = 2` bits | `rho_AB` with `I(A:B) < 2` — correlation leaked to the environment |
| pure, maximally entangled joint state | mixed `rho_AB` (fidelity < 1), reconstructed from finite shots |

Where `05/03` measured how far one qubit fell from a pure target, here you measure how much *shared information* two qubits lost — entanglement made quantitative as a number of bits.

## Your turn

- **Try the other Bell state.** Build the `|01> + |10>` variant with `prep.h(0); prep.cx(0, 1); prep.x(1)` and re-run. Its ideal `I(A:B)` is also 2 bits, so check whether the device's value drops by a similar amount — the entanglement loss should track the gate count, not which Bell state you chose.
- **Flip `USE_HARDWARE = True`** (with credentials set per `05/01`) and re-run. The gap below 2 bits is now the real chip's entanglement loss, not a model's.
- **Raise the shots and split the gap.** Re-run at higher shot counts (say `8192`, then `16384`) and watch the *finite-shot* part of the gap shrink while the *systematic* decoherence part stays put. This is the same statistical-versus-systematic split you met on the Born rule in `05/02` — only now it acts on an information-geometric quantity instead of a single probability.

## What you built

- A full two-qubit tomography run on the primitive path: a Bell pair prepared on the device, nine Pauli-basis circuits transpiled and sampled with `SamplerV2`, and the joint state `rho_AB` reconstructed with `density_from_counts`.
- The quantum mutual information `I(A:B)` read off that reconstructed state, the first information-geometric quantity of this module measured on a live device.
- A direct sighting of entanglement loss: `I(A:B)` falls below the ideal 2 bits, with the gap split honestly between decoherence (the dominant, irreducible part) and finite-shot reconstruction bias (the part that shrinks with more shots).
- A new dictionary row tying the ideal `I(A:B) = 2` bits to the device's diminished value.

## What's next

Next you measure **geometry** on real hardware: the **Bures distance** between the ideal and device states, turning the fidelity gap of `05/03` into a genuine metric and setting up the optimal-transport geometry of module 04 against a live chip.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 11 (eq. 11.78, quantum mutual information), Cambridge University Press (2010).
- M. M. Wilde, *Quantum Information Theory*, ch. 11, Cambridge University Press (2017), doi:10.1017/9781316809976.

**Previous:** `notebooks/05_RealQuantumHardware/03_tomography_and_fidelity_on_real_hardware.ipynb` · **Next:** `notebooks/05_RealQuantumHardware/05_geometry_on_real_hardware_bures.ipynb`